In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os, glob, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
import pandas as pd, glob, os
ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"

csv_path = glob.glob(ROOT + "/**/*.csv", recursive=True)
print(csv_path)

meta = pd.read_csv(csv_path[0])
print(meta.shape)
print(meta.columns.tolist())
display(meta.head(5))
print(meta.dtypes)

# useful summaries (these skip any column that doesn't exist)
for c in meta.columns:
    if meta[c].nunique() < 15:
        print(f"\n{c}:\n", meta[c].value_counts())

['/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo/meta.csv']
(25988, 8)
['Unnamed: 0', 'alert_idx', 'center_idx', 'date', 'sampling_type', 'state', 'file', 'close_set']


,Unnamed: 0,alert_idx,center_idx,date,sampling_type,state,file,close_set
0,0,0,0,2021-08-21,positive,PA,0000000_2021-08-21.pt,train
1,1,1,1,2020-09-20,positive,AM,0000001_2020-09-20.pt,train
2,2,2,2,2020-08-08,boundary,PA,0000002_2020-08-08.pt,validation
3,3,4,3,2020-07-24,positive,PA,0000003_2020-07-24.pt,train
4,4,5,4,2020-08-06,boundary,PA,0000004_2020-08-06.pt,train


Unnamed: 0        int64
alert_idx         int64
center_idx        int64
date             object
sampling_type    object
state            object
file             object
close_set        object
dtype: object

sampling_type:
 sampling_type
positive       8712
oldDeforest    7557
forest         5472
boundary       2294
herbaceous     1780
agriculture     152
shrubs           21
Name: count, dtype: int64

state:
 state
PA    11604
MT     3264
AM     2901
RO     2875
AC     2329
RR     1457
MA     1048
AP      390
TO      120
Name: count, dtype: int64

close_set:
 close_set
train         15625
test           5251
validation     5112
Name: count, dtype: int64


In [4]:
import torch, pandas as pd, glob, os
ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"
meta = pd.read_csv(f"{ROOT}/meta.csv")

# where do the .pt files live?
pt_files = glob.glob(ROOT + "/**/*.pt", recursive=True)
print(len(pt_files), pt_files[:2])

# open one positive + one forest sample
for st in ["positive", "forest"]:
    fn = meta[meta.sampling_type == st].file.iloc[0]
    p = [x for x in pt_files if x.endswith(fn)][0]
    obj = torch.load(p, map_location="cpu", weights_only=False)
    print("\n", st, type(obj))
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(" ", k, getattr(v, "shape", v if not hasattr(v, "__len__") else type(v)),
                  getattr(v, "dtype", ""))
            if hasattr(v, "unique") and v.ndim <= 3: print("    unique:", v.unique()[:10])
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj): print(" ", i, getattr(v, "shape", type(v)), getattr(v, "dtype", ""))
    else:
        print(getattr(obj, "shape", None))

25989 ['/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo/close_stats.pt', '/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo/Samples/0000000_2021-08-21.pt']

 positive <class 'dict'>
  image_dates <class 'list'> 
  label_dates <class 'list'> 
  image torch.Size([31, 2, 48, 48]) torch.float32
  label torch.Size([2, 48, 48]) torch.int64
    unique: tensor([0, 1])

 forest <class 'dict'>
  image_dates <class 'list'> 
  label_dates <class 'list'> 
  image torch.Size([31, 2, 48, 48]) torch.float32
  label torch.Size([2, 48, 48]) torch.int64
    unique: tensor([0])


In [7]:
import torch, numpy as np, pandas as pd
ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"
meta = pd.read_csv(f"{ROOT}/meta.csv")

# 1) normalization stats file
st = torch.load(f"{ROOT}/close_stats.pt", map_location="cpu", weights_only=False)
print(type(st))
print({k: (getattr(v, "shape", v)) for k, v in st.items()} if isinstance(st, dict) else st)

# 2) label channels + dates on 6 positive samples
for fn in meta[meta.sampling_type == "positive"].file.iloc[:6]:
    d = torch.load(f"{ROOT}/Samples/{fn}", map_location="cpu", weights_only=False)
    lab = d["label"]
    print(fn,
          "| ch0 sum:", int(lab[0].sum()), "| ch1 sum:", int(lab[1].sum()),
          "| identical:", bool((lab[0] == lab[1]).all()),
          "| label_dates:", d["label_dates"],
          "| n image dates:", len(d["image_dates"]),
          "| first/last:", d["image_dates"][0], d["image_dates"][-1])

# 3) is the number of frames always 31?
lens = [torch.load(f"{ROOT}/Samples/{f}", map_location='cpu', weights_only=False)["image"].shape[0]
        for f in meta.file.sample(200, random_state=0)]
print("frames per sample:", set(lens))

<class 'dict'>
{'mean': torch.Size([2]), 'std': torch.Size([2]), 'min': torch.Size([]), 'max': torch.Size([])}
0000000_2021-08-21.pt | ch0 sum: 0 | ch1 sum: 789 | identical: False | label_dates: [datetime.date(2020, 8, 27), datetime.date(2021, 8, 21)] | n image dates: 31 | first/last: 2020-08-27 2021-09-03
0000001_2020-09-20.pt | ch0 sum: 0 | ch1 sum: 653 | identical: False | label_dates: [datetime.date(2019, 9, 25), datetime.date(2020, 9, 20)] | n image dates: 25 | first/last: 2019-09-25 2020-10-13
0000003_2020-07-24.pt | ch0 sum: 0 | ch1 sum: 577 | identical: False | label_dates: [datetime.date(2019, 7, 31), datetime.date(2020, 7, 24)] | n image dates: 30 | first/last: 2019-07-31 2020-08-06
0000006_2021-07-24.pt | ch0 sum: 0 | ch1 sum: 716 | identical: False | label_dates: [datetime.date(2020, 8, 3), datetime.date(2021, 7, 24)] | n image dates: 31 | first/last: 2020-08-03 2021-08-10
0000009_2021-08-02.pt | ch0 sum: 0 | ch1 sum: 1067 | identical: False | label_dates: [datetime.date(20

In [6]:
import numpy as np, random, torch, pandas as pd, os
from torch.utils.data import Dataset, DataLoader

ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"
meta = pd.read_csv(f"{ROOT}/meta.csv")

T_FIXED  = 24
LABEL_CH = 1

stats = torch.load(f"{ROOT}/close_stats.pt", map_location="cpu", weights_only=False)
BAND_MEAN = stats["mean"].numpy().astype(np.float32)   # (2,)
BAND_STD  = stats["std"].numpy().astype(np.float32)    # (2,)
print("mean:", BAND_MEAN, "std:", BAND_STD, "min/max:", stats["min"].item(), stats["max"].item())

class BraDD(Dataset):
    def __init__(self, df, augment=False):
        self.files = df.file.tolist(); self.aug = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        d = torch.load(f"{ROOT}/Samples/{self.files[i]}", map_location="cpu", weights_only=False)
        x = np.nan_to_num(d["image"].float().numpy())                 # (T, 2, 48, 48), T varies
        y = (d["label"][LABEL_CH].numpy() > 0).astype(np.float32)     # (48, 48)
        idx = np.linspace(0, x.shape[0]-1, T_FIXED).astype(int)       # evenly spaced frames
        x = (x[idx] - BAND_MEAN[None, :, None, None]) / (BAND_STD[None, :, None, None] + 1e-6)
        x = x.reshape(-1, *x.shape[-2:])                              # (T*2, 48, 48)
        if self.aug:
            if random.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
            if random.random() < .5: x, y = x[:, ::-1, :], y[::-1, :]
        return torch.from_numpy(x.copy()).float(), torch.from_numpy(y.copy())[None].float()

tr_df = meta[meta.close_set == "train"].reset_index(drop=True)
va_df = meta[meta.close_set == "validation"].reset_index(drop=True)
te_df = meta[meta.close_set == "test"].reset_index(drop=True)

xs, ys = BraDD(tr_df)[0]
IN_CH = xs.shape[0]
print("x:", xs.shape, "| y:", ys.shape, "| deforested px:", int(ys.sum()), "| IN_CH:", IN_CH)

train_dl = DataLoader(BraDD(tr_df, True), batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(BraDD(va_df),       batch_size=64, shuffle=False, num_workers=4)
test_dl  = DataLoader(BraDD(te_df),       batch_size=64, shuffle=False, num_workers=4)

mean: [ -8.87772  -15.201087] std: [2.9824772 3.1364543] min/max: -64.78938293457031 22.154420852661133
x: torch.Size([48, 48, 48]) | y: torch.Size([1, 48, 48]) | deforested px: 789 | IN_CH: 48


In [8]:
rows = []
for st, g in meta.groupby("sampling_type"):
    for fn in g.file.sample(min(30, len(g)), random_state=0):
        d = torch.load(f"{ROOT}/Samples/{fn}", map_location="cpu", weights_only=False)
        rows.append((st, int(d["label"][0].sum()), int(d["label"][1].sum())))
chk = pd.DataFrame(rows, columns=["type", "ch0_px", "ch1_px"])
display(chk.groupby("type").agg(ch0_mean=("ch0_px", "mean"), ch1_mean=("ch1_px", "mean"),
                                ch1_pct_nonzero=("ch1_px", lambda s: (s > 0).mean())))

,ch0_mean,ch1_mean,ch1_pct_nonzero
type,,,
agriculture,0.000000,0.000000,0.0
boundary,5.866667,592.566667,1.0
forest,21.566667,0.000000,0.0
herbaceous,0.000000,0.000000,0.0
oldDeforest,35.233333,0.000000,0.0
positive,12.300000,928.466667,1.0
shrubs,0.000000,0.000000,0.0


In [9]:
T_FIXED = 31          # all frames are used, so IN_CH = 31*2 = 62
LABEL_CH = None       # set to 0 / 1 after the check above; None = merge both channels

def load_sample(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    x = d["image"].float().numpy()                       # (31, 2, 48, 48)
    lab = d["label"].numpy()                             # (2, 48, 48)
    y = lab.max(0) if LABEL_CH is None else lab[LABEL_CH]
    return x, (y > 0).astype(np.float32)

In [11]:
rows = []
for st, g in meta.groupby("sampling_type"):
    for fn in g.file.sample(min(30, len(g)), random_state=0):
        d = torch.load(f"{ROOT}/Samples/{fn}", map_location="cpu", weights_only=False)
        rows.append((st, int(d["label"][0].sum()), int(d["label"][1].sum())))
chk = pd.DataFrame(rows, columns=["type", "ch0_px", "ch1_px"])
display(chk.groupby("type").agg(ch0_mean=("ch0_px", "mean"), ch1_mean=("ch1_px", "mean"),
                                ch1_pct_nonzero=("ch1_px", lambda s: (s > 0).mean())))

,ch0_mean,ch1_mean,ch1_pct_nonzero
type,,,
agriculture,0.000000,0.000000,0.0
boundary,5.866667,592.566667,1.0
forest,21.566667,0.000000,0.0
herbaceous,0.000000,0.000000,0.0
oldDeforest,35.233333,0.000000,0.0
positive,12.300000,928.466667,1.0
shrubs,0.000000,0.000000,0.0


In [10]:
import numpy as np, random, torch, pandas as pd, os
from torch.utils.data import Dataset, DataLoader

ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"
meta = pd.read_csv(f"{ROOT}/meta.csv")

T_FIXED  = 24
LABEL_CH = 1

stats = torch.load(f"{ROOT}/close_stats.pt", map_location="cpu", weights_only=False)
BAND_MEAN = stats["mean"].numpy().astype(np.float32)   # (2,)
BAND_STD  = stats["std"].numpy().astype(np.float32)    # (2,)
print("mean:", BAND_MEAN, "std:", BAND_STD, "min/max:", stats["min"].item(), stats["max"].item())

class BraDD(Dataset):
    def __init__(self, df, augment=False):
        self.files = df.file.tolist(); self.aug = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        d = torch.load(f"{ROOT}/Samples/{self.files[i]}", map_location="cpu", weights_only=False)
        x = np.nan_to_num(d["image"].float().numpy())                 # (T, 2, 48, 48), T varies
        y = (d["label"][LABEL_CH].numpy() > 0).astype(np.float32)     # (48, 48)
        idx = np.linspace(0, x.shape[0]-1, T_FIXED).astype(int)       # evenly spaced frames
        x = (x[idx] - BAND_MEAN[None, :, None, None]) / (BAND_STD[None, :, None, None] + 1e-6)
        x = x.reshape(-1, *x.shape[-2:])                              # (T*2, 48, 48)
        if self.aug:
            if random.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
            if random.random() < .5: x, y = x[:, ::-1, :], y[::-1, :]
        return torch.from_numpy(x.copy()).float(), torch.from_numpy(y.copy())[None].float()

tr_df = meta[meta.close_set == "train"].reset_index(drop=True)
va_df = meta[meta.close_set == "validation"].reset_index(drop=True)
te_df = meta[meta.close_set == "test"].reset_index(drop=True)

xs, ys = BraDD(tr_df)[0]
IN_CH = xs.shape[0]
print("x:", xs.shape, "| y:", ys.shape, "| deforested px:", int(ys.sum()), "| IN_CH:", IN_CH)

train_dl = DataLoader(BraDD(tr_df, True), batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(BraDD(va_df),       batch_size=64, shuffle=False, num_workers=4)
test_dl  = DataLoader(BraDD(te_df),       batch_size=64, shuffle=False, num_workers=4)

mean: [ -8.87772  -15.201087] std: [2.9824772 3.1364543] min/max: -64.78938293457031 22.154420852661133
x: torch.Size([48, 48, 48]) | y: torch.Size([1, 48, 48]) | deforested px: 789 | IN_CH: 48


In [12]:
@torch.no_grad()
def predict_all(model, dl, thr=0.5):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append((torch.sigmoid(model(x.to(device))) > thr).cpu().bool()); Y.append(y.bool())
    return torch.cat(P), torch.cat(Y)

def iou_f1(p, y):
    tp = (p & y).sum().item(); fp = (p & ~y).sum().item(); fn = (~p & y).sum().item()
    return dict(IoU=tp/(tp+fp+fn+1e-9), F1=2*tp/(2*tp+fp+fn+1e-9), n=len(p))

results = {}
for name, M in models.items():
    m = M(IN_CH).to(device); m.load_state_dict(torch.load(f"{name}_best.pt"))
    P, Y = predict_all(m, test_dl)
    results[name] = dict(
        overall=iou_f1(P, Y),
        by_type={t: iou_f1(P[ix], Y[ix]) for t, ix in te_df.groupby("sampling_type").indices.items()},
        by_state={s: iou_f1(P[ix], Y[ix]) for s, ix in te_df.groupby("state").indices.items()},
    )

display(pd.DataFrame({n: {t: v["IoU"] for t, v in r["by_type"].items()} for n, r in results.items()}).round(3))
display(pd.DataFrame({n: {s: v["IoU"] for s, v in r["by_state"].items()} for n, r in results.items()}).round(3))

NameError: name 'models' is not defined

In [13]:
import torch, torch.nn as nn, torch.nn.functional as F

def cbr(i, o):
    """Conv -> BatchNorm -> ReLU, the basic building brick."""
    return [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True)]

# ---------- 1) UNet ----------
class DoubleConv(nn.Sequential):
    def __init__(s, i, o):
        super().__init__(*cbr(i, o), *cbr(o, o))

class UNet(nn.Module):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__()
        s.enc = nn.ModuleList(); c = in_ch
        for k in f:
            s.enc.append(DoubleConv(c, k)); c = k
        s.pool = nn.MaxPool2d(2)
        s.bott = DoubleConv(f[-1], f[-1] * 2)
        s.up  = nn.ModuleList()
        s.dec = nn.ModuleList()
        for k in reversed(f):
            s.up.append(nn.ConvTranspose2d(k * 2, k, 2, 2))
            s.dec.append(DoubleConv(k * 2, k))
        s.out = nn.Conv2d(f[0], out_ch, 1)

    def forward(s, x):
        skips = []
        for e in s.enc:
            x = e(x); skips.append(x); x = s.pool(x)
        x = s.bott(x)
        for u, d, sk in zip(s.up, s.dec, reversed(skips)):
            x = d(torch.cat([u(x), sk], 1))
        return s.out(x)

# ---------- 2) Residual UNet ----------
class ResBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.body = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o))
        s.short = (nn.Sequential(nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o))
                   if i != o else nn.Identity())
    def forward(s, x):
        return F.relu(s.body(x) + s.short(x))   # output = block(x) + x

class ResUNet(UNet):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__(in_ch, out_ch, f)      # same layout as UNet...
        s.enc = nn.ModuleList(); c = in_ch      # ...but swap the blocks for residual ones
        for k in f:
            s.enc.append(ResBlock(c, k)); c = k
        s.bott = ResBlock(f[-1], f[-1] * 2)
        s.dec = nn.ModuleList([ResBlock(k * 2, k) for k in reversed(f)])

# ---------- 3) SegNet ----------
class SegNet(nn.Module):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__()
        s.enc = nn.ModuleList(); s.dec = nn.ModuleList(); c = in_ch
        for k in f:
            s.enc.append(nn.Sequential(*cbr(c, k), *cbr(k, k))); c = k
        for j in range(len(f) - 1, -1, -1):
            k = f[j]; o = f[max(j - 1, 0)]
            s.dec.append(nn.Sequential(*cbr(k, k), *cbr(k, o)))
        s.pool   = nn.MaxPool2d(2, 2, return_indices=True)  # remembers the "winner" positions
        s.unpool = nn.MaxUnpool2d(2, 2)                      # puts values back at those positions
        s.out = nn.Conv2d(f[0], out_ch, 1)

    def forward(s, x):
        idxs, sizes = [], []
        for e in s.enc:
            x = e(x); sizes.append(x.shape)
            x, i = s.pool(x); idxs.append(i)
        for d in s.dec:
            x = s.unpool(x, idxs.pop(), output_size=sizes.pop())
            x = d(x)
        return s.out(x)

models = {"UNet": UNet, "ResUNet": ResUNet, "SegNet": SegNet}

# ---------- sanity check ----------
for n, M in models.items():
    net = M(48)
    out = net(torch.randn(2, 48, 48, 48))   # fake batch: 2 samples, 48 channels, 48x48
    print(f"{n:8s} output {tuple(out.shape)} | {sum(p.numel() for p in net.parameters())/1e6:.2f}M params")

UNet     output (2, 1, 48, 48) | 7.78M params
ResUNet  output (2, 1, 48, 48) | 8.13M params
SegNet   output (2, 1, 48, 48) | 2.37M params


In [14]:
import torch, torch.nn.functional as F

def dice_loss(logits, y, eps=1.0):
    """Computed over the whole batch, so empty-mask samples don't create noisy gradients."""
    p = torch.sigmoid(logits)                       # logits -> probability (0 to 1)
    inter = (p * y).sum()                           # overlap with the real mask
    return 1 - (2 * inter + eps) / (p.sum() + y.sum() + eps)

def loss_fn(logits, y):
    return F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)

@torch.no_grad()
def evaluate(model, dl, thr=0.5):
    """Runs the model on a dataloader and returns overall pixel-level scores."""
    model.eval()
    tp = fp = fn = tn = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        p = (torch.sigmoid(model(x)) > thr).float() # probability -> 0/1 prediction
        tp += (p * y).sum().item()                  # predicted yes, truly yes
        fp += (p * (1 - y)).sum().item()            # predicted yes, truly no
        fn += ((1 - p) * y).sum().item()            # predicted no, truly yes
        tn += ((1 - p) * (1 - y)).sum().item()      # predicted no, truly no
    e = 1e-9
    prec, rec = tp / (tp + fp + e), tp / (tp + fn + e)
    return dict(IoU=tp / (tp + fp + fn + e),
                F1=2 * prec * rec / (prec + rec + e),
                Precision=prec, Recall=rec,
                Acc=(tp + tn) / (tp + tn + fp + fn + e))

# ---------- quick test on an untrained model (should run without errors) ----------
net = UNet(48).to(device)
print("untrained UNet on val:", {k: round(v, 3) for k, v in evaluate(net, val_dl).items()})

untrained UNet on val: {'IoU': 0.0, 'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0, 'Acc': 0.844}


In [15]:
import torch, time

def train(name, epochs=30, lr=1e-3):
    model = models[name](IN_CH).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    scaler = torch.amp.GradScaler("cuda")
    best, hist = 0.0, []
    for ep in range(epochs):
        t0 = time.time()
        model.train(); tot = 0.0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.autocast("cuda"):
                loss = loss_fn(model(x), y)          # forward pass + score
            scaler.scale(loss).backward()            # work out how to improve
            scaler.step(opt); scaler.update()        # adjust the weights
            tot += loss.item()
        sched.step()
        m = evaluate(model, val_dl)                  # check on unseen validation data
        hist.append(dict(epoch=ep + 1, loss=tot / len(train_dl), **m))
        if m["IoU"] > best:
            best = m["IoU"]
            torch.save(model.state_dict(), f"{name}_best.pt")
        print(f"{name} ep{ep+1:02d} | loss {tot/len(train_dl):.4f} | val IoU {m['IoU']:.4f} "
              f"F1 {m['F1']:.4f} P {m['Precision']:.3f} R {m['Recall']:.3f} | {time.time()-t0:.0f}s")
    return hist

# ---------- smoke test: 2 epochs only ----------
_ = train("UNet", epochs=2)

UNet ep01 | loss 0.9525 | val IoU 0.3731 F1 0.5435 P 0.608 R 0.491 | 2678s
UNet ep02 | loss 0.8283 | val IoU 0.4009 F1 0.5724 P 0.660 R 0.505 | 29s


In [16]:
import pandas as pd

histories = {}
for name in ["UNet", "ResUNet", "SegNet"]:
    print(f"\n===== Training {name} =====")
    torch.manual_seed(42)                       # same random start recipe for each model
    h = train(name, epochs=30)
    histories[name] = pd.DataFrame(h)
    histories[name].to_csv(f"{name}_history.csv", index=False)   # saved right away
    print(f"{name} best val IoU: {histories[name].IoU.max():.4f} "
          f"(epoch {histories[name].IoU.idxmax() + 1})")


===== Training UNet =====
UNet ep01 | loss 0.9660 | val IoU 0.3671 F1 0.5370 P 0.454 R 0.658 | 30s
UNet ep02 | loss 0.8452 | val IoU 0.4168 F1 0.5884 P 0.537 R 0.650 | 29s
UNet ep03 | loss 0.8167 | val IoU 0.4197 F1 0.5913 P 0.628 R 0.559 | 29s
UNet ep04 | loss 0.8028 | val IoU 0.4304 F1 0.6018 P 0.600 R 0.604 | 29s
UNet ep05 | loss 0.7907 | val IoU 0.4210 F1 0.5925 P 0.679 R 0.525 | 29s
UNet ep06 | loss 0.7797 | val IoU 0.4357 F1 0.6069 P 0.569 R 0.650 | 30s
UNet ep07 | loss 0.7701 | val IoU 0.4463 F1 0.6172 P 0.609 R 0.625 | 29s
UNet ep08 | loss 0.7634 | val IoU 0.4297 F1 0.6011 P 0.654 R 0.556 | 29s
UNet ep09 | loss 0.7523 | val IoU 0.3628 F1 0.5324 P 0.738 R 0.416 | 29s
UNet ep10 | loss 0.7478 | val IoU 0.4492 F1 0.6199 P 0.649 R 0.593 | 29s
UNet ep11 | loss 0.7404 | val IoU 0.4451 F1 0.6161 P 0.659 R 0.578 | 29s
UNet ep12 | loss 0.7328 | val IoU 0.4341 F1 0.6054 P 0.677 R 0.547 | 29s
UNet ep13 | loss 0.7303 | val IoU 0.4572 F1 0.6275 P 0.628 R 0.627 | 29s
UNet ep14 | loss 0.7202 

In [17]:
import pandas as pd

rows = {}
for name, M in models.items():
    net = M(IN_CH).to(device)
    net.load_state_dict(torch.load(f"{name}_best.pt"))
    r = evaluate(net, test_dl)
    r["Params(M)"] = sum(p.numel() for p in net.parameters()) / 1e6
    rows[name] = r

test_table = pd.DataFrame(rows).T.round(4)
display(test_table)
test_table.to_csv("test_results.csv")

,IoU,F1,Precision,Recall,Acc,Params(M)
UNet,0.4732,0.6424,0.6639,0.6223,0.8875,7.7760
ResUNet,0.4687,0.6383,0.6659,0.6129,0.8871,8.1286
SegNet,0.4640,0.6339,0.6431,0.6249,0.8827,2.3673


In [18]:
@torch.no_grad()
def predict_all(model, dl, thr=0.5):
    """Returns binary predictions and ground truth for the whole test set (same order as te_df)."""
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append((torch.sigmoid(model(x.to(device))) > thr).cpu().bool())
        Y.append(y.bool())
    return torch.cat(P), torch.cat(Y)

def group_scores(P, Y):
    tp = (P & Y).sum().item(); fp = (P & ~Y).sum().item(); fn = (~P & Y).sum().item()
    nan = float("nan")
    return dict(n=len(P),
                IoU=tp / (tp + fp + fn) if (tp + fp + fn) > 0 else nan,
                Recall=tp / (tp + fn) if (tp + fn) > 0 else nan,
                FP_px_per_sample=fp / len(P))

preds = {}                                   # keep predictions, we reuse them in the next steps
for name, M in models.items():
    net = M(IN_CH).to(device)
    net.load_state_dict(torch.load(f"{name}_best.pt"))
    preds[name] = predict_all(net, test_dl)

by_type = {}
for name, (P, Y) in preds.items():
    by_type[name] = {t: group_scores(P[ix], Y[ix])
                     for t, ix in te_df.groupby("sampling_type").indices.items()}

pos_types = ["positive", "boundary"]
neg_types = ["forest", "oldDeforest", "herbaceous", "agriculture", "shrubs"]

print("=== Samples with deforestation: IoU / Recall ===")
display(pd.DataFrame({n: {t: by_type[n][t]["IoU"] for t in pos_types} for n in models}).round(3))
display(pd.DataFrame({n: {t: by_type[n][t]["Recall"] for t in pos_types} for n in models}).round(3))

print("=== Samples WITHOUT deforestation: false-alarm pixels per sample (lower is better) ===")
tbl = pd.DataFrame({n: {t: by_type[n][t]["FP_px_per_sample"] for t in neg_types} for n in models}).round(1)
tbl.insert(0, "n_samples", [by_type["UNet"][t]["n"] for t in neg_types])
display(tbl)

=== Samples with deforestation: IoU / Recall ===


,UNet,ResUNet,SegNet
positive,0.523,0.514,0.515
boundary,0.413,0.401,0.407


,UNet,ResUNet,SegNet
positive,0.633,0.625,0.634
boundary,0.558,0.544,0.571


=== Samples WITHOUT deforestation: false-alarm pixels per sample (lower is better) ===


,n_samples,UNet,ResUNet,SegNet
forest,1068,30.5,23.9,37.0
oldDeforest,1508,85.3,75.5,90.3
herbaceous,381,10.2,6.9,10.2
agriculture,33,0.0,0.0,0.0
shrubs,5,0.0,0.0,0.0


In [19]:
import numpy as np, pandas as pd, torch

@torch.no_grad()
def get_probs(model, dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append(torch.sigmoid(model(x.to(device))).cpu())
        Y.append(y.bool())
    return torch.cat(P), torch.cat(Y)

def iou_at(P, Y, thr):
    p = P > thr
    tp = (p & Y).sum().item(); fp = (p & ~Y).sum().item(); fn = (~p & Y).sum().item()
    return tp / (tp + fp + fn + 1e-9)

thrs = np.arange(0.20, 0.81, 0.05)
best_thr, rows = {}, {}
for name, M in models.items():
    net = M(IN_CH).to(device)
    net.load_state_dict(torch.load(f"{name}_best.pt"))
    Pv, Yv = get_probs(net, val_dl)
    Pt, Yt = get_probs(net, test_dl)
    val_curve = [iou_at(Pv, Yv, t) for t in thrs]
    b = float(thrs[int(np.argmax(val_curve))])          # chosen on validation only
    best_thr[name] = b
    rows[name] = {"best_thr": round(b, 2),
                  "val IoU @0.5": iou_at(Pv, Yv, 0.5), "val IoU @best": max(val_curve),
                  "test IoU @0.5": iou_at(Pt, Yt, 0.5), "test IoU @best": iou_at(Pt, Yt, b)}

display(pd.DataFrame(rows).T.round(4))

,best_thr,val IoU @0.5,val IoU @best,test IoU @0.5,test IoU @best
UNet,0.35,0.4714,0.4745,0.4732,0.4777
ResUNet,0.35,0.4668,0.4706,0.4687,0.4717
SegNet,0.40,0.4625,0.4638,0.4640,0.4652


In [20]:
from dataclasses import dataclass
import numpy as np, pandas as pd, random, torch, time
from torch.utils.data import Dataset, DataLoader

@dataclass
class Cfg:
    T: int = 24
    frames: str = "even"       # "even" = evenly spaced, "last" = final T frames
    rot_aug: bool = False      # extra 90-degree rotations
    lr: float = 1e-3
    wd: float = 1e-4
    epochs: int = 15
    bs: int = 32
    seed: int = 42

class BraDD2(Dataset):
    def __init__(self, df, cfg, augment=False):
        self.files, self.cfg, self.aug = df.file.tolist(), cfg, augment
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        c = self.cfg
        d = torch.load(f"{ROOT}/Samples/{self.files[i]}", map_location="cpu", weights_only=False)
        x = np.nan_to_num(d["image"].float().numpy())                 # (n, 2, 48, 48)
        y = (d["label"][1].numpy() > 0).astype(np.float32)
        n = x.shape[0]
        if c.frames == "even":
            idx = np.linspace(0, n - 1, c.T).astype(int)
        else:                                                         # last T frames
            idx = np.clip(np.arange(n - c.T, n), 0, n - 1)
        x = (x[idx] - BAND_MEAN[None, :, None, None]) / (BAND_STD[None, :, None, None] + 1e-6)
        x = np.clip(x, -5, 5).reshape(-1, 48, 48)
        if self.aug:
            if random.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
            if random.random() < .5: x, y = x[:, ::-1, :], y[::-1, :]
            if c.rot_aug:
                k = random.randint(0, 3)
                x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (0, 1))
        return torch.from_numpy(x.copy()).float(), torch.from_numpy(y.copy())[None].float()

def run_trial(tag, cfg, name="UNet"):
    torch.manual_seed(cfg.seed); random.seed(cfg.seed); np.random.seed(cfg.seed)
    tr = DataLoader(BraDD2(tr_df, cfg, True), batch_size=cfg.bs, shuffle=True, num_workers=4, pin_memory=True)
    va = DataLoader(BraDD2(va_df, cfg),       batch_size=64, num_workers=4)
    model = models[name](2 * cfg.T).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.epochs)
    scaler = torch.amp.GradScaler("cuda")
    hist, best = [], 0.0
    for ep in range(cfg.epochs):
        t0 = time.time(); model.train(); tot = 0.0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.autocast("cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()
        sched.step()
        m = evaluate(model, va)
        hist.append(m["IoU"])
        if m["IoU"] > best:
            best = m["IoU"]; torch.save(model.state_dict(), f"{name}_{tag}.pt")
        print(f"[{tag}] ep{ep+1:02d} loss {tot/len(tr):.4f} val IoU {m['IoU']:.4f} ({time.time()-t0:.0f}s)")
    return dict(tag=tag, best_val_IoU=best, last3_mean=float(np.mean(hist[-3:])))

trial_cfgs = {
    "even24": Cfg(T=24, frames="even"),
    "even16": Cfg(T=16, frames="even"),
    "even32": Cfg(T=32, frames="even"),
    "last24": Cfg(T=24, frames="last"),
}
trial_results = []
for tag, cfg in trial_cfgs.items():
    trial_results.append(run_trial(tag, cfg))
    pd.DataFrame(trial_results).to_csv("trials_input.csv", index=False)   # saved after each trial

display(pd.DataFrame(trial_results).round(4))

[even24] ep01 loss 0.9538 val IoU 0.3981 (29s)
[even24] ep02 loss 0.8420 val IoU 0.4159 (29s)
[even24] ep03 loss 0.8127 val IoU 0.4187 (29s)
[even24] ep04 loss 0.7991 val IoU 0.4214 (29s)
[even24] ep05 loss 0.7847 val IoU 0.4350 (29s)
[even24] ep06 loss 0.7710 val IoU 0.4432 (29s)
[even24] ep07 loss 0.7604 val IoU 0.4375 (29s)
[even24] ep08 loss 0.7514 val IoU 0.4529 (29s)
[even24] ep09 loss 0.7373 val IoU 0.4039 (29s)
[even24] ep10 loss 0.7305 val IoU 0.4587 (29s)
[even24] ep11 loss 0.7205 val IoU 0.4570 (29s)
[even24] ep12 loss 0.7122 val IoU 0.4518 (29s)
[even24] ep13 loss 0.7063 val IoU 0.4574 (29s)
[even24] ep14 loss 0.6959 val IoU 0.4518 (29s)
[even24] ep15 loss 0.6985 val IoU 0.4663 (29s)
[even16] ep01 loss 0.9472 val IoU 0.3567 (27s)
[even16] ep02 loss 0.8613 val IoU 0.4163 (27s)
[even16] ep03 loss 0.8359 val IoU 0.3709 (27s)
[even16] ep04 loss 0.8102 val IoU 0.3953 (26s)
[even16] ep05 loss 0.7969 val IoU 0.4198 (26s)
[even16] ep06 loss 0.7829 val IoU 0.4423 (27s)
[even16] ep07

,tag,best_val_IoU,last3_mean
0,even24,0.4663,0.4585
1,even16,0.4582,0.4562
2,even32,0.4682,0.4676
3,last24,0.3984,0.3865


In [1]:
import os, numpy as np, pandas as pd, torch
from multiprocessing import Pool

ROOT = "/kaggle/input/datasets/sangatichapla/bradd-s1/BraDD-S1TS_zenodo"
meta = pd.read_csv(f"{ROOT}/meta.csv")
stats = torch.load(f"{ROOT}/close_stats.pt", map_location="cpu", weights_only=False)
MEAN = stats["mean"].numpy().astype(np.float32)[None, :, None, None]
STD  = stats["std"].numpy().astype(np.float32)[None, :, None, None]
T_CACHE = 32
N = len(meta)

def read_one(i):
    torch.set_num_threads(1)
    d = torch.load(f"{ROOT}/Samples/{meta.file.iloc[i]}", map_location="cpu", weights_only=False)
    x = np.nan_to_num(d["image"].float().numpy())
    idx = np.linspace(0, x.shape[0] - 1, T_CACHE).astype(int)
    x = np.clip((x[idx] - MEAN) / (STD + 1e-6), -5, 5).astype(np.float16)
    y = (d["label"][1].numpy() > 0).astype(np.uint8)
    return i, x, y

OUT = "/kaggle/working"
X = np.lib.format.open_memmap(f"{OUT}/X32.npy", mode="w+", dtype=np.float16,
                              shape=(N, T_CACHE, 2, 48, 48))
Y = np.zeros((N, 48, 48), dtype=np.uint8)

with Pool(8) as pool:
    for k, (i, x, y) in enumerate(pool.imap_unordered(read_one, range(N), chunksize=64)):
        X[i] = x; Y[i] = y
        if k % 2000 == 0: print(f"{k}/{N}", flush=True)

X.flush(); np.save(f"{OUT}/Y.npy", Y)
meta.to_csv(f"{OUT}/meta.csv", index=False)
print("done:", X.shape, Y.shape, "| deforested px share:", round(float(Y.mean()), 3))

0/25988
2000/25988
4000/25988
6000/25988
8000/25988
10000/25988
12000/25988
14000/25988
16000/25988
18000/25988
20000/25988
22000/25988
24000/25988
done: (25988, 32, 2, 48, 48) (25988, 48, 48) | deforested px share: 0.159


In [3]:
!pip -q install -U kaggle

import os, json
from kaggle_secrets import UserSecretsClient

os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
USER = "sangatichapla"

PACK = "/kaggle/working/pack"
os.makedirs(PACK, exist_ok=True)
for f in ["X32.npy", "Y.npy", "meta.csv"]:
    src = f"/kaggle/working/{f}"
    if os.path.exists(src):                                # skip files already moved
        os.rename(src, f"{PACK}/{f}")

json.dump({"title": "bradd-cache32",
           "id": f"{USER}/bradd-cache32",
           "licenses": [{"name": "other"}]},
          open(f"{PACK}/dataset-metadata.json", "w"))

!ls -lh /kaggle/working/pack
!kaggle datasets create -p /kaggle/working/pack --dir-mode skip

total 7.2G
-rw-r--r-- 1 root root   96 Oct 31 10:13 dataset-metadata.json
-rw-r--r-- 1 root root 1.5M Oct 31 09:44 meta.csv
-rw-r--r-- 1 root root 7.2G Oct 31 09:44 X32.npy
-rw-r--r-- 1 root root  58M Oct 31 09:44 Y.npy
Starting upload for file X32.npy
100%|██████████████████████████████████████| 7.14G/7.14G [01:39<00:00, 76.6MB/s]
Upload successful: X32.npy (7GB)
Starting upload for file meta.csv
100%|██████████████████████████████████████| 1.47M/1.47M [00:00<00:00, 3.63MB/s]
Upload successful: meta.csv (1MB)
Starting upload for file Y.npy
100%|██████████████████████████████████████| 57.1M/57.1M [00:01<00:00, 57.8MB/s]
Upload successful: Y.npy (57MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/sangatichapla/bradd-cache32


In [4]:
import glob, os
hits = glob.glob("/kaggle/input/**/X32.npy", recursive=True)
print(hits)
D = os.path.dirname(hits[0])
print("D =", D)
print(os.listdir(D))

['/kaggle/input/datasets/sangatichapla/bradd-cache32/X32.npy']
D = /kaggle/input/datasets/sangatichapla/bradd-cache32
['X32.npy', 'Y.npy', 'meta.csv']


In [2]:
import glob, os, time, random
import numpy as np, pandas as pd, torch

device = "cuda"
D = os.path.dirname(glob.glob("/kaggle/input/**/X32.npy", recursive=True)[0])

t0 = time.time()
meta  = pd.read_csv(f"{D}/meta.csv")
Y_all = torch.from_numpy(np.load(f"{D}/Y.npy"))        # (N, 48, 48) uint8
X_all = torch.from_numpy(np.load(f"{D}/X32.npy"))      # (N, 32, 2, 48, 48) float16, ~7.7 GB in RAM
print("loaded", tuple(X_all.shape), f"in {time.time()-t0:.0f}s")

idx = {s: np.where(meta.close_set.values == s)[0] for s in ["train", "validation", "test"]}
print({s: len(v) for s, v in idx.items()})

def frame_ids(T):
    return np.linspace(0, 31, T).astype(int)

def get_batch(ids, T=32, augment=False):
    x = X_all[ids][:, frame_ids(T)]
    y = Y_all[ids]
    x = x.reshape(len(ids), 2 * T, 48, 48).to(device, non_blocking=True).float()
    y = y[:, None].to(device, non_blocking=True).float()
    if augment:
        if random.random() < .5: x, y = x.flip(3), y.flip(3)
        if random.random() < .5: x, y = x.flip(2), y.flip(2)
    return x, y

def batches(split, bs, T=32, shuffle=False, augment=False):
    ids = idx[split].copy()
    if shuffle: np.random.shuffle(ids)
    for i in range(0, len(ids), bs):
        yield get_batch(np.sort(ids[i:i + bs]), T, augment)

x, y = next(batches("train", 32, T=32, shuffle=True, augment=True))
print("x:", tuple(x.shape), "| y:", tuple(y.shape), "| x mean/std:",
      round(x.mean().item(), 3), round(x.std().item(), 3), "| deforested share:", round(y.mean().item(), 3))

loaded (25988, 32, 2, 48, 48) in 83s
{'train': 15625, 'validation': 5112, 'test': 5251}
x: (32, 64, 48, 48) | y: (32, 1, 48, 48) | x mean/std: 0.045 1.002 | deforested share: 0.124


In [3]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd, random, time

# ---------------- models (same as before) ----------------
def cbr(i, o): return [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True)]
class DoubleConv(nn.Sequential):
    def __init__(s, i, o): super().__init__(*cbr(i, o), *cbr(o, o))
class UNet(nn.Module):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__()
        s.enc = nn.ModuleList(); c = in_ch
        for k in f: s.enc.append(DoubleConv(c, k)); c = k
        s.pool = nn.MaxPool2d(2); s.bott = DoubleConv(f[-1], f[-1] * 2)
        s.up = nn.ModuleList(); s.dec = nn.ModuleList()
        for k in reversed(f):
            s.up.append(nn.ConvTranspose2d(k * 2, k, 2, 2)); s.dec.append(DoubleConv(k * 2, k))
        s.out = nn.Conv2d(f[0], out_ch, 1)
    def forward(s, x):
        sk = []
        for e in s.enc: x = e(x); sk.append(x); x = s.pool(x)
        x = s.bott(x)
        for u, d, k in zip(s.up, s.dec, reversed(sk)): x = d(torch.cat([u(x), k], 1))
        return s.out(x)
class ResBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.body = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
                               nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o))
        s.short = nn.Sequential(nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o)) if i != o else nn.Identity()
    def forward(s, x): return F.relu(s.body(x) + s.short(x))
class ResUNet(UNet):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__(in_ch, out_ch, f)
        s.enc = nn.ModuleList(); c = in_ch
        for k in f: s.enc.append(ResBlock(c, k)); c = k
        s.bott = ResBlock(f[-1], f[-1] * 2)
        s.dec = nn.ModuleList([ResBlock(k * 2, k) for k in reversed(f)])
class SegNet(nn.Module):
    def __init__(s, in_ch, out_ch=1, f=(32, 64, 128, 256)):
        super().__init__()
        s.enc = nn.ModuleList(); s.dec = nn.ModuleList(); c = in_ch
        for k in f: s.enc.append(nn.Sequential(*cbr(c, k), *cbr(k, k))); c = k
        for j in range(len(f) - 1, -1, -1):
            s.dec.append(nn.Sequential(*cbr(f[j], f[j]), *cbr(f[j], f[max(j - 1, 0)])))
        s.pool = nn.MaxPool2d(2, 2, return_indices=True); s.unpool = nn.MaxUnpool2d(2, 2)
        s.out = nn.Conv2d(f[0], out_ch, 1)
    def forward(s, x):
        idxs, sizes = [], []
        for e in s.enc:
            x = e(x); sizes.append(x.shape); x, i = s.pool(x); idxs.append(i)
        for d in s.dec: x = d(s.unpool(x, idxs.pop(), output_size=sizes.pop()))
        return s.out(x)
models = {"UNet": UNet, "ResUNet": ResUNet, "SegNet": SegNet}

# ---------------- losses ----------------
def focal_loss(z, y, alpha=0.75, gamma=1.0):
    """The paper's alpha-balanced focal loss (their Eq. 1), computed from logits."""
    z = z.float(); p = torch.sigmoid(z)
    pos = alpha * y * (1 - p) ** gamma * F.logsigmoid(z)
    neg = (1 - alpha) * (1 - y) * p ** gamma * F.logsigmoid(-z)
    return -(pos + neg).mean()

def bce_dice(z, y):
    z = z.float(); p = torch.sigmoid(z)
    return F.binary_cross_entropy_with_logits(z, y) + 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)

# ---------------- evaluation: pixel AND patch level ----------------
@torch.no_grad()
def predict(model, split, T=32, bs=256):
    model.eval(); P, Yl = [], []
    for x, y in batches(split, bs, T):
        with torch.autocast("cuda"): z = model(x)
        P.append(torch.sigmoid(z.float()).cpu()); Yl.append(y.cpu())
    return torch.cat(P), torch.cat(Yl) > 0.5

def prf(tp, fp, fn):
    return dict(IoU=tp / (tp + fp + fn + 1e-9), P=tp / (tp + fp + 1e-9), R=tp / (tp + fn + 1e-9))

def scores(P, Y, thr):
    p = P > thr
    pix = prf((p & Y).sum().item(), (p & ~Y).sum().item(), (~p & Y).sum().item())
    pp, py = p.flatten(1).any(1), Y.flatten(1).any(1)      # patch positive = at least 1 positive pixel
    pat = prf((pp & py).sum().item(), (pp & ~py).sum().item(), (~pp & py).sum().item())
    return pix, pat

THRS = np.arange(0.2, 0.81, 0.05)
def best_threshold(P, Y):
    ious = [scores(P, Y, t)[0]["IoU"] for t in THRS]
    return float(THRS[int(np.argmax(ious))]), max(ious)

# ---------------- training ----------------
def train(name, tag, loss_kind="focal", alpha=0.75, gamma=1.0, epochs=30, lr=1e-3, wd=1e-4,
          T=32, bs=32, seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = models[name](2 * T).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    scaler = torch.amp.GradScaler("cuda")
    lf = (lambda z, y: focal_loss(z, y, alpha, gamma)) if loss_kind == "focal" else bce_dice
    best, best_thr, hist = 0.0, 0.5, []
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = 0.0; nb = 0
        for x, y in batches("train", bs, T, shuffle=True, augment=True):
            opt.zero_grad()
            with torch.autocast("cuda"): loss = lf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); nb += 1
        sched.step()
        Pv, Yv = predict(model, "validation", T)
        thr, iou = best_threshold(Pv, Yv)                    # select epoch on best-threshold val IoU
        hist.append(dict(epoch=ep + 1, loss=tot / nb, val_IoU=iou, thr=thr))
        if iou > best:
            best, best_thr = iou, thr
            torch.save({"state": model.state_dict(), "thr": thr, "T": T}, f"{tag}.pt")
        print(f"[{tag}] ep{ep+1:02d} loss {tot/nb:.4f} | val IoU {iou:.4f} @thr {thr:.2f} | {time.time()-t0:.0f}s")
    pd.DataFrame(hist).to_csv(f"{tag}_history.csv", index=False)
    return best, best_thr

def test_report(name, tag, T=32):
    ck = torch.load(f"{tag}.pt"); m = models[name](2 * T).to(device); m.load_state_dict(ck["state"])
    P, Y = predict(m, "test", T)
    pix, pat = scores(P, Y, ck["thr"])
    print(f"{tag} | thr {ck['thr']:.2f}")
    print("  pixel:", {k: round(v, 4) for k, v in pix.items()})
    print("  patch:", {k: round(v, 4) for k, v in pat.items()})
    return pix, pat

# ---------------- run: UNet + the paper's focal loss ----------------
train("UNet", "UNet_focal")
pix, pat = test_report("UNet", "UNet_focal")

[UNet_focal] ep01 loss 0.0803 | val IoU 0.3985 @thr 0.60 | 20s
[UNet_focal] ep02 loss 0.0728 | val IoU 0.4207 @thr 0.55 | 17s
[UNet_focal] ep03 loss 0.0712 | val IoU 0.4235 @thr 0.50 | 17s
[UNet_focal] ep04 loss 0.0697 | val IoU 0.4210 @thr 0.55 | 17s
[UNet_focal] ep05 loss 0.0683 | val IoU 0.4164 @thr 0.55 | 17s
[UNet_focal] ep06 loss 0.0674 | val IoU 0.4272 @thr 0.65 | 17s
[UNet_focal] ep07 loss 0.0668 | val IoU 0.4514 @thr 0.60 | 17s
[UNet_focal] ep08 loss 0.0655 | val IoU 0.4465 @thr 0.60 | 17s
[UNet_focal] ep09 loss 0.0652 | val IoU 0.4501 @thr 0.50 | 17s
[UNet_focal] ep10 loss 0.0642 | val IoU 0.4581 @thr 0.55 | 17s
[UNet_focal] ep11 loss 0.0635 | val IoU 0.4467 @thr 0.55 | 17s
[UNet_focal] ep12 loss 0.0631 | val IoU 0.4493 @thr 0.50 | 17s
[UNet_focal] ep13 loss 0.0625 | val IoU 0.4606 @thr 0.55 | 17s
[UNet_focal] ep14 loss 0.0618 | val IoU 0.4498 @thr 0.55 | 17s
[UNet_focal] ep15 loss 0.0616 | val IoU 0.4673 @thr 0.55 | 17s
[UNet_focal] ep16 loss 0.0609 | val IoU 0.4643 @thr 0.5

In [4]:
@torch.no_grad()
def predict_tta(model, split, T=32, bs=256):
    model.eval(); P, Yl = [], []
    for x, y in batches(split, bs, T):
        acc = 0
        for fl in [None, (2,), (3,), (2, 3)]:
            xi = x if fl is None else x.flip(fl)
            with torch.autocast("cuda"): o = torch.sigmoid(model(xi).float())
            acc = acc + (o if fl is None else o.flip(fl))
        P.append((acc / 4).cpu()); Yl.append(y.cpu())
    return torch.cat(P), torch.cat(Yl) > 0.5

ck = torch.load("UNet_focal.pt"); m = UNet(64).to(device); m.load_state_dict(ck["state"])
Pv, Yv = predict_tta(m, "validation"); Pt, Yt = predict_tta(m, "test")
torch.save({"Pv": Pv.half(), "Pt": Pt.half()}, "probs_UNet_focal.pt")    # for ensembling later

# 1) pixel level: TTA effect (threshold re-tuned on validation)
thr, v_iou = best_threshold(Pv, Yv)
pix, _ = scores(Pt, Yt, thr)
print(f"PIXEL  TTA | thr {thr:.2f} | val IoU {v_iou:.4f} | test IoU {pix['IoU']:.4f} "
      f"(P {pix['P']:.3f} R {pix['R']:.3f})   [no TTA was 0.4739]")

# 2) patch level: choose (pixel thr, min positive pixels k) on validation
def patch_scores(P, Y, t, k):
    pp = (P > t).flatten(1).sum(1) >= k
    py = Y.flatten(1).any(1)
    return prf((pp & py).sum().item(), (pp & ~py).sum().item(), (~pp & py).sum().item())

grid = [(t, k) for t in THRS for k in [1, 2, 5, 10, 20, 50, 100]]
vals = [patch_scores(Pv, Yv, t, k)["IoU"] for t, k in grid]
bt, bk = grid[int(np.argmax(vals))]
print(f"PATCH  best rule on val: pixel thr {bt:.2f}, min pixels k={bk} | val IoU {max(vals):.4f}")
print("PATCH  test, old rule (k=1):", {a: round(b, 4) for a, b in patch_scores(Pt, Yt, thr, 1).items()})
print("PATCH  test, tuned rule    :", {a: round(b, 4) for a, b in patch_scores(Pt, Yt, bt, bk).items()})

PIXEL  TTA | thr 0.55 | val IoU 0.4791 | test IoU 0.4786 (P 0.623 R 0.674)   [no TTA was 0.4739]
PATCH  best rule on val: pixel thr 0.60, min pixels k=20 | val IoU 0.6910
PATCH  test, old rule (k=1): {'IoU': 0.6937, 'P': 0.7728, 'R': 0.8715}
PATCH  test, tuned rule    : {'IoU': 0.7024, 'P': 0.8198, 'R': 0.8307}


In [5]:
import os, torch, numpy as np, pandas as pd

# ---------- TTA predictions (4 flips averaged) ----------
@torch.no_grad()
def predict_tta(model, split, T=32, bs=256):
    model.eval(); P = []
    for x, y in batches(split, bs, T):
        acc = 0
        for fl in [None, (2,), (3,), (2, 3)]:
            o = model(x if fl is None else x.flip(fl))
            o = torch.sigmoid(o.float())
            acc = acc + (o if fl is None else o.flip(fl))
        P.append((acc / 4).cpu())
    return torch.cat(P)

# ---------- patch rule: patch = deforested if >= k pixels above threshold t ----------
KS = [1, 2, 5, 10, 20, 50, 100]
def patch_eval(P, Y, t, k):
    pp = (P > t).flatten(1).sum(1) >= k
    py = Y.flatten(1).any(1)
    return prf((pp & py).sum().item(), (pp & ~py).sum().item(), (~pp & py).sum().item())

def patch_best(P, Y):
    best = (-1, 0.5, 1)
    for t in THRS:
        cnt = (P > t).flatten(1).sum(1); py = Y.flatten(1).any(1)
        for k in KS:
            pp = cnt >= k
            r = prf((pp & py).sum().item(), (pp & ~py).sum().item(), (~pp & py).sum().item())
            if r["IoU"] > best[0]: best = (r["IoU"], float(t), k)
    return best[1], best[2]

def full_eval(Pv, Yv, Pt, Yt):
    thr, viou = best_threshold(Pv, Yv)            # all choices made on validation
    pix, _ = scores(Pt, Yt, thr)
    t, k = patch_best(Pv, Yv)
    pat = patch_eval(Pt, Yt, t, k)
    return dict(pix_thr=round(thr, 2), val_pix_IoU=viou, test_pix_IoU=pix["IoU"],
                test_pix_P=pix["P"], test_pix_R=pix["R"], patch_t=round(t, 2), patch_k=k,
                test_patch_IoU=pat["IoU"], test_patch_P=pat["P"], test_patch_R=pat["R"])

Yv = (Y_all[idx["validation"]] > 0)[:, None]
Yt = (Y_all[idx["test"]] > 0)[:, None]

# ---------- train all runs (resumable) ----------
RUNS = [(m, s) for s in [42, 1, 2] for m in ["UNet", "ResUNet", "SegNet"]]
rows = []
for name, seed in RUNS:
    tag = f"{name}_focal_s{seed}"; pf = f"probs_{tag}.pt"
    if not os.path.exists(pf):
        train(name, tag, seed=seed)
        ck = torch.load(f"{tag}.pt"); m = models[name](64).to(device); m.load_state_dict(ck["state"])
        torch.save({"Pv": predict_tta(m, "validation").half(),
                    "Pt": predict_tta(m, "test").half()}, pf)
    pr = torch.load(pf)
    rows.append(dict(run=tag, model=name, seed=seed,
                     **full_eval(pr["Pv"].float(), Yv, pr["Pt"].float(), Yt)))
    pd.DataFrame(rows).to_csv("runs_summary.csv", index=False)     # saved after every run
    print(rows[-1], flush=True)

# ---------- ensembles: average the probabilities of several runs ----------
def ensemble(tags):
    ps = [torch.load(f"probs_{t}.pt") for t in tags]
    return (sum(p["Pv"].float() for p in ps) / len(ps), sum(p["Pt"].float() for p in ps) / len(ps))

groups = {f"{m} x3 seeds": [f"{m}_focal_s{s}" for s in [42, 1, 2]] for m in ["UNet", "ResUNet", "SegNet"]}
groups["ALL 9 models"] = [f"{m}_focal_s{s}" for m in ["UNet", "ResUNet", "SegNet"] for s in [42, 1, 2]]
ens_rows = []
for label, tags in groups.items():
    Pv, Pt = ensemble(tags)
    ens_rows.append(dict(group=label, **full_eval(Pv, Yv, Pt, Yt)))
ens = pd.DataFrame(ens_rows); ens.to_csv("ensemble_summary.csv", index=False)

runs = pd.DataFrame(rows)
print("\n=== Single runs: mean +- std over 3 seeds ===")
print(runs.groupby("model")[["test_pix_IoU", "test_patch_IoU", "test_patch_P", "test_patch_R"]]
      .agg(["mean", "std"]).round(4))
print("\n=== Ensembles ===")
print(ens[["group", "test_pix_IoU", "test_pix_P", "test_pix_R",
           "test_patch_IoU", "test_patch_P", "test_patch_R"]].round(4).to_string(index=False))

[UNet_focal_s42] ep01 loss 0.0803 | val IoU 0.3985 @thr 0.60 | 18s
[UNet_focal_s42] ep02 loss 0.0728 | val IoU 0.4207 @thr 0.55 | 18s
[UNet_focal_s42] ep03 loss 0.0712 | val IoU 0.4235 @thr 0.50 | 18s
[UNet_focal_s42] ep04 loss 0.0697 | val IoU 0.4210 @thr 0.55 | 17s
[UNet_focal_s42] ep05 loss 0.0683 | val IoU 0.4164 @thr 0.55 | 18s
[UNet_focal_s42] ep06 loss 0.0674 | val IoU 0.4272 @thr 0.65 | 17s
[UNet_focal_s42] ep07 loss 0.0668 | val IoU 0.4514 @thr 0.60 | 17s
[UNet_focal_s42] ep08 loss 0.0655 | val IoU 0.4465 @thr 0.60 | 17s
[UNet_focal_s42] ep09 loss 0.0652 | val IoU 0.4501 @thr 0.50 | 17s
[UNet_focal_s42] ep10 loss 0.0642 | val IoU 0.4581 @thr 0.55 | 17s
[UNet_focal_s42] ep11 loss 0.0635 | val IoU 0.4467 @thr 0.55 | 17s
[UNet_focal_s42] ep12 loss 0.0631 | val IoU 0.4493 @thr 0.50 | 17s
[UNet_focal_s42] ep13 loss 0.0625 | val IoU 0.4606 @thr 0.55 | 17s
[UNet_focal_s42] ep14 loss 0.0618 | val IoU 0.4498 @thr 0.55 | 17s
[UNet_focal_s42] ep15 loss 0.0616 | val IoU 0.4673 @thr 0.55 |

In [7]:
import glob, os
for pat in ["*_focal_s*.pt", "probs_*_focal_s*.pt", "*.csv"]:
    fs = sorted(glob.glob(f"/kaggle/working/{pat}"))
    print(pat, "->", len(fs), "files")

*_focal_s*.pt -> 18 files
probs_*_focal_s*.pt -> 9 files
*.csv -> 13 files


In [8]:
!pip -q install -U kaggle

import os, json, shutil, glob
from kaggle_secrets import UserSecretsClient

os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
USER = "sangatichapla"

PACK = "/kaggle/working/results_pack"
os.makedirs(PACK, exist_ok=True)
for pat in ["*_focal_s*.pt", "probs_*_focal_s*.pt", "*.csv"]:
    for f in glob.glob(f"/kaggle/working/{pat}"):
        shutil.copy(f, PACK)                       # copy, so the originals stay if the upload fails

json.dump({"title": "bradd-results", "id": f"{USER}/bradd-results",
           "licenses": [{"name": "other"}]},
          open(f"{PACK}/dataset-metadata.json", "w"))

!ls -lh /kaggle/working/results_pack | head -40
!kaggle datasets create -p /kaggle/working/results_pack --dir-mode skip

total 622M
-rw-r--r-- 1 root root   96 Nov  1 18:39 dataset-metadata.json
-rw-r--r-- 1 root root  759 Nov  1 18:39 ensemble_summary.csv
-rw-r--r-- 1 root root 1.5M Nov  1 18:39 meta.csv
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_ResUNet_focal_s1.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_ResUNet_focal_s2.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_ResUNet_focal_s42.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_SegNet_focal_s1.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_SegNet_focal_s2.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_SegNet_focal_s42.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_UNet_focal_s1.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_UNet_focal_s2.pt
-rw-r--r-- 1 root root  46M Nov  1 18:39 probs_UNet_focal_s42.pt
-rw-r--r-- 1 root root 1.9K Nov  1 18:39 ResUNet_focal_s1_history.csv
-rw-r--r-- 1 root root  32M Nov  1 18:39 ResUNet_focal_s1.pt
-rw-r--r-- 1 root root 1.9K Nov  1 18:39 ResUNet_focal_s2_history.csv
-rw-r--r-- 1 root r